In [2]:
import matplotlib.pyplot as plt
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.animation as animation

from robot.robot import Robot

%matplotlib widget
%load_ext autoreload
%autoreload 2

os.environ['KMP_DUPLICATE_LIB_OK']='True' # To prevent the kernel from dying.

RuntimeError: 'widget' is not a recognised GUI loop or backend name


<div class="alert alert-success">
    <h3>Task: Check Code</h3>
    <p> Examine the implementation of the <code>Robot</code> class in the folder <code>exercise_code/robot/robot.py</code>. The <code>Robot</code> object takes a number of inputs like the length of the arms, the intial angle, constraints for the angles the robot can move in, etc., as Python Lists.</p>
    <p>Make sure that the number of arm lengths passed to the Robot initializer match the number of angles and the constraints.</p>
    
</div>

In [ ]:
armLengths    = []   # enter desired values for arm length of each section of the robot arm separated by commas
initialAngles = [] # enter desired values for intial angles (degrees counter clockwise from the +ve x-axis) 
maximumAngles = [] # constraints for each section
minimumAngles = [] # constraints for each section
a = Robot(lengths=armLengths, initAngles=initialAngles,
          maxAngles=maximumAngles, minAngles=minimumAngles)

In [ ]:
print(a)
a.plotRobot()

<div class="alert alert-info">
    <h3>Task: Implement</h3>
    <p> To optimize a system, we need to have an objective function for it that has its global minimum at the desired target point. In this case the objective function is the distance of the robot arm's tip to the target point. This function is dependent on the Target Point, and the individual angles of the Robots's arms</p>
    <p>In <code>robot/robot.py</code> complete the implementation of the <code>distToTarget</code> function. Use the Euclidean Distance formula to compue the distance.</p>
    
</div>

$$d(\mathbf{p}, \mathbf{q}) = \sqrt{ \sum_{i=1}^{2} (q_i - p_i)^2 }$$



<div class="alert alert-info">
    <h3>Task: Implement</h3>
    <p>To optimize a function with the stochastic gradient descent method, we need to use the derivative of the function w.r.t. the optimization paramaters, here the angles of the robot's arms.</p>    
    <p>In <code>robot/robot.py</code> complete the implementation of the <code>gradient</code> function. You can manually derive the derivative w.r.t. each angle of the robot arm, or you can use the Finite Diffference Method to approximate the derivative. </p>
    <small>Hint: As the distance is always going to be a positive value with a minima at zero, the derivative of the ssquare of the distance can also be used.</small>
    <p>Execute the cell below and check the output against your own calculations.</p>
</div>

FDM: $$f^{\prime}(\mathbf{x})=\frac{f(\mathbf{x}+h) - f(\mathbf{x})}{h}$$


In [ ]:
a.gradient()

### Set the target point below

In [1]:
target = [0.5 ,2]

<div class="alert alert-info">
    <h3>Task: Implement</h3>
    <p>In <code>robot/robot.py</code> complete the implementation of the <code>Solve</code> function. Use the Explicit Forward Euler method to compute the Angle update.</p>

</div>

Explicit Forward Euler:
$$\mathbf{x}^{(k+1)}=\mathbf{x}^{(k)}-\alpha_{k} \cdot \nabla f\left(\mathbf{x}^{(k)}\right)$$

In [ ]:
a.Solve(targetPoint=target)

### Use the cell below to generate an animation of the robot converging to the set target point.

The update function plots the current position of the robot arm and the history of its tip location for each frame of the animation. It constantly updates the tip location history arrays (historyX, historyY), which is why they are defined as global variables and not limited as internal variables of the update function. The list of arrays fo the joint locations is also defined as a global variable. Also used as global variables are the line and trace objects generated by the plot function. They are Line2D objects and the update function simply updates the data these objects are plotting by using the set_xdata and set_ydata methods of the Line2D objects.

The FuncAnimation function from matplotlib uses this update function to render the animation fram by frame. The FuncAnimation function, as its name might imply, is used to generate animations from a function that takes an integer value as an input (the frame number) and generates a plot for that frame of the animation.

In [ ]:
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(autoscale_on=False, xlim=(-10, 10), ylim=(-10, 10))
ax.set_aspect('equal')
ax.grid()
joints = a.poseCompute(a.solnAngles[0])
line, = ax.plot(joints[:,0], joints[:,1], 'o-', lw=2, label='Robot') # arms of the robot
trace, = ax.plot(joints[-1,0], joints[-1,1], '.-', lw=1, ms=2, label='Path trace') # path of the robot arm's tip
ax.scatter(target[0], target[1],  s=23, marker='x', label='Target Point')
ax.legend()

historyX = []
historyY = []

def update(frame):
    """
    updates the plot with the next position for the animation function
    """
    joints = a.poseCompute(a.solnAngles[frame])

    if frame == 0:
        historyX.clear()
        historyY.clear()

    historyX.append(joints[-1,0])
    historyY.append(joints[-1,1])

    line.set_xdata(joints[:,0])
    line.set_ydata(joints[:,1])
    trace.set_xdata(historyX)
    trace.set_ydata(historyY)

    return line, trace

historyX.clear()
historyY.clear()
ani = animation.FuncAnimation(fig, update, len(a.solnAngles), repeat=True,
                                blit=True)
plt.show()
ani.save("SGDRobot.gif", writer="pillow")

<div class="alert alert-success">
    <h3>Success!</h3>
    <p>Check the final position of the robot arm below. How close were you to the target point.</p>
    <p>Play with the Solver settings like the alpha value and the distance threshold to see if you can get any closer</p>
</div>

In [ ]:
a.plotRobot()